# PHASE 2: Data Cleaning & Quality Checks
**Traceability**
- Issue ID: #2 Data Cleaning & Quality Checks

## 1. Objectives
- Remove constant sensors with zero or near-zero variance to reduce noise and dimensionality.
- Handle outliers using robust statistical methods without breaking temporal continuity.
- Verify sensor scales to inform future normalization choices.

### 2.1 Import Libraries & Configure Paths
We import necessary libraries and define paths for raw and processed data.

In [3]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# ── Reproducibility Config ──────────────────────────────────────────────
np.random.seed(42)

# ── Global Config ────────────────────────────────────────────────────────
DATA_DIR = Path('../data')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

COL_NAMES = (
    ['unit_number', 'time_cycles'] +
    ['setting_1', 'setting_2', 'setting_3'] +
    [f's_{i}' for i in range(1, 22)]   # s_1 ... s_21
)

### 2.2 Constant Sensor Detection
Identify sensors that do not change significantly over time. These sensors do not provide useful information for predictive maintenance.

In [4]:
def detect_constant_sensors(df, threshold=0.01):
    """Identify sensors with standard deviation below a threshold."""
    sensor_cols = [f's_{i}' for i in range(1, 22)]
    sensor_std = df[sensor_cols].std()
    constant_sensors = sensor_std[sensor_std < threshold].index.tolist()
    return constant_sensors

### 2.3 Outlier Handling (Clipping)
Instead of removing rows, which would break the temporal sequence of engine cycles, we clip outliers to the 1st and 99th percentiles.

In [5]:
def handle_outliers(df, sensor_cols):
    """Clip outliers to 1st and 99th percentiles (preserve temporal continuity)."""
    df_clipped = df.copy()
    for col in sensor_cols:
        q1, q3 = df_clipped[col].quantile(0.01), df_clipped[col].quantile(0.99)
        df_clipped[col] = df_clipped[col].clip(lower=q1, upper=q3)
    return df_clipped

### 2.4 Scale Assessment
Analyze the range and variation of each sensor to understand the scaling requirements for machine learning models.

In [6]:
def scale_check(df, sensor_cols):
    """Compute descriptive statistics to assess sensor scales."""
    stats = df[sensor_cols].describe().T
    stats['range'] = stats['max'] - stats['min']
    stats['cv'] = stats['std'] / (stats['mean'] + 1e-9)
    return stats[['mean', 'std', 'min', 'max', 'range', 'cv']].sort_values('range', ascending=False)

### 2.5 Execution: Clean & Save
Load the data, remove constant sensors, clip outliers, and save the cleaned datasets.

In [7]:
# 1. Load Data
df_train = pd.read_csv(DATA_DIR / 'train_FD001.txt', sep=r'\s+', header=None, index_col=False, names=COL_NAMES)
df_test = pd.read_csv(DATA_DIR / 'test_FD001.txt', sep=r'\s+', header=None, index_col=False, names=COL_NAMES)
print(f"✅ Data loaded: Train {df_train.shape}, Test {df_test.shape}")

# 2. Identify and Drop Constant Sensors
constant_sensors = detect_constant_sensors(df_train)
print(f"🔍 Constant sensors to drop (std < 0.01): {constant_sensors}")

df_train = df_train.drop(columns=constant_sensors)
df_test = df_test.drop(columns=constant_sensors)

remaining_sensors = [c for c in df_train.columns if c.startswith('s_')]
print(f"✅ Remaining sensors ({len(remaining_sensors)}): {remaining_sensors}")

# 3. Handle Outliers (Clipping)
df_train = handle_outliers(df_train, remaining_sensors)
df_test = handle_outliers(df_test, remaining_sensors)
print(f"✅ Outliers clipped (1st/99th percentiles). Row counts preserved.")

# 4. Scale Check
stats = scale_check(df_train, remaining_sensors)
print("\n  SENSOR SCALE SUMMARY (Sorted by Range):")
print(stats.round(3))

# 5. Save Cleaned Data
df_train.to_csv(PROCESSED_DIR / 'train_cleaned.csv', index=False)
df_test.to_csv(PROCESSED_DIR / 'test_cleaned.csv', index=False)
print(f"\n✅ Cleaned data saved to {PROCESSED_DIR}")

✅ Data loaded: Train (20631, 26), Test (13096, 26)
🔍 Constant sensors to drop (std < 0.01): ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']
✅ Remaining sensors (14): ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']
✅ Outliers clipped (1st/99th percentiles). Row counts preserved.

  SENSOR SCALE SUMMARY (Sorted by Range):
          mean     std       min       max    range     cv
s_9   9065.037  20.812  9036.350  9155.239  118.889  0.002
s_14  8143.580  17.937  8115.416  8220.020  104.604  0.002
s_4   1408.927   8.906  1391.883  1431.517   39.634  0.006
s_3   1590.520   6.038  1577.613  1605.871   28.258  0.004
s_17   393.208   1.533   390.000   397.000    7.000  0.004
s_7    553.368   0.874   551.160   555.070    3.910  0.002
s_12   521.414   0.729   519.550   522.800    3.250  0.001
s_2    642.681   0.493   641.650   643.930    2.280  0.001
s_11    47.541   0.264    47.050    48.210    1.160  0.006
s_20    38.816   0.178 